# muse-lab runner (Kaggle)

このノートブックは GitHub の `muse-lab` リポジトリを clone し、指定ブランチの実験コードを GPU で実行します。
結果は最後のセルの出力（`METRICS_JSON:` 行）に集約されます。

**回し方**: 上から順に全セルを実行するだけ。条件を変えたいときは次の CONFIG セルだけ書き換えます。

## 1. CONFIG（ここだけ書き換える）

In [ ]:
BRANCH = "main"  # 実験ブランチ名に変えると、そのブランチのコードで回る
TRAIN_ARGS = "--epochs 30 --lr 0.01 --seed 42"  # experiments/train.py への引数
REPO_URL = "https://github.com/c2vt8h92hw-star/muse-lab.git"

## 2. リポジトリを取得

In [ ]:
import os, subprocess, sys
WORKDIR = "/kaggle/working"
repo_dir = os.path.join(WORKDIR, "muse-lab")
if os.path.exists(repo_dir):
    subprocess.run(["rm", "-rf", repo_dir], check=True)
subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, repo_dir], check=True)
os.chdir(repo_dir)
print("cloned:", repo_dir, "| branch:", BRANCH)
print(os.listdir("."))

## 3. GPU確認

In [ ]:
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## 4. 実験を実行

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "experiments/train.py"] + TRAIN_ARGS.split(), capture_output=True, text=True)
print(r.stdout)
print(r.stderr[-2000:] if r.returncode != 0 else "", end="")

## 5. 結果の読み方

最後の `METRICS_JSON: {...}` 行が結果の要約です（train_acc / test_acc / device / seconds など）。
次の実験では CONFIG の `TRAIN_ARGS` を変えるか、GitHub 側でコードを更新してから再実行します。